In [1]:
import librosa
import matplotlib.pyplot as plt
import librosa.display
import os
import numpy as np
from tensorflow.keras import layers
from IPython.display import Audio
import wave
import pandas as pd
import tensorflow as tf
import cv2
import pandas as pd
import matplotlib.pyplot as plt

2025-02-16 10:58:43.050927: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-16 10:58:43.956913: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
audio_fp = "/home/alien/Git/DATA/LibriStutterData/LibriStutter_16kHz/"
transcript_fp = "/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/"

In [3]:
fp = "/home/alien/Git/XSpeech/data_processing/output16.csv"

In [4]:
df = pd.read_csv(fp)
df

,filepath,results,new_filepath
0,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
1,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
2,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
3,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
4,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
...,...,...,...
3908,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
3909,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
3910,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
3911,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...


In [5]:
print(df.iloc[0,1])

/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0027.txt


In [6]:
from scipy.io import wavfile

def get_sampling_rate(filename):
    sampling_rate, _ = wavfile.read(filename)
    return sampling_rate

# Example usage
filename = df.iloc[0,0]
sampling_rate = get_sampling_rate(filename)
print(f'Sampling Rate: {sampling_rate} Hz')

Sampling Rate: 16000 Hz


In [7]:
print(df.iloc[:,1][0])

/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0027.txt


In [8]:
df_list_fps = df.iloc[:,0].tolist()
df_list_text = df.iloc[:,1].tolist()

In [9]:
df_list_text

['/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0027.txt',
 '/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0034.txt',
 '/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0055.txt',
 '/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0042.txt',
 '/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0048.txt',
 '/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0044.txt',
 '/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0013.txt',
 '/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0004.txt',
 '/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0019.txt',
 '/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcripts/103/1240/103-1240-0020.txt',
 '/home/alien/Git/DATA/LibriStutterData/LibriStutter Transcr

In [10]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")


from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="English", task="transcribe")

/home/alien/Programming/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/alien/Programming/env/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [11]:
input_str = "Hello, World!"
labels = tokenizer(input_str).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")


Input:                 Hello, World!
Decoded w/ special:    <|startoftranscript|><|en|><|transcribe|><|notimestamps|>Hello, World!<|endoftext|>
Decoded w/out special: Hello, World!
Are equal:             True


In [12]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="English", task="transcribe")

/home/alien/Programming/env/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [13]:
# Load a .wav file
def load_wav(filename, sr=16000):
    y, sr = librosa.load(filename, sr=sr)
    return y

In [14]:
y = load_wav(df.iloc[0,0])

In [15]:
y.shape

(246486,)

# convert to dict

In [16]:
def prepare_dataset(fp, text_fp): #given one
    # read
    with open(text_fp, 'r') as file:
        text = file.read()
    # compute log-Mel input features from input audio array 
    input_features = feature_extractor(load_wav(fp), sampling_rate=16000).input_features[0]

    # encode target text to label ids 
    labels = tokenizer(text).input_ids
    print(text)
    return input_features, labels

In [17]:
len(df_list_text)

3913

In [18]:
len(df_list_fps)

3913

In [19]:
batch = []
# each dict 
for i in range(len(df_list_fps)):
    f, l = prepare_dataset(df_list_fps[i], df_list_text[i])
    batch.append({
        "input_features": f,
        "labels": l
    })

they were three plates late so the Marilla must be expecting someone home with Matthew 2T but the dishes for everyday dishes and there was only Crabapple preserves and one kind of cake so that the expected company could not be any particular company
he was actually stricken dumb for 5 it was on supposable that Marilla was making fun of her but misses Rachel was almost forced to supposed are you in Earnest Marilla she demanded when voice returned to her yes of course
it doesn't really seem as if I must be dreaming while I'm sorry for that poor young one and no mistake Matthew Marilla don't know anything about children and they'll expect him to be wiser and steadier than his own grandfather
so in the end we decided to ask mrs. Spencer to pick us out one when went over to get her little girl we heard last week she was going so we sent her word by Richard Spencer's Folks at Carmody to bring us a smart likely boy of about 10 or 11 we decided that would be the best date
this job's comforting

In [20]:
print(batch[0])

{'input_features': array([[ 0.24611044,  0.46459454,  0.48897463, ..., -0.91613007,
        -0.91613007, -0.91613007],
       [ 0.45975995,  0.46088415,  0.4555158 , ..., -0.91613007,
        -0.91613007, -0.91613007],
       [ 0.35326147,  0.2833082 ,  0.26900327, ..., -0.91613007,
        -0.91613007, -0.91613007],
       ...,
       [-0.53035176, -0.65966356, -0.70018995, ..., -0.91613007,
        -0.91613007, -0.91613007],
       [-0.5704118 , -0.8895104 , -0.91613007, ..., -0.91613007,
        -0.91613007, -0.91613007],
       [-0.5767702 , -0.91613007, -0.91613007, ..., -0.91613007,
        -0.91613007, -0.91613007]], dtype=float32), 'labels': [50258, 50259, 50359, 50363, 13162, 645, 1045, 14231, 3469, 370, 264, 2039, 5291, 1633, 312, 9650, 1580, 1280, 365, 12434, 568, 51, 457, 264, 10814, 337, 7429, 10814, 293, 456, 390, 787, 383, 5305, 21316, 1183, 9054, 293, 472, 733, 295, 5908, 370, 300, 264, 5176, 2237, 727, 406, 312, 604, 1729, 2237, 50257]}


In [21]:
f, l = prepare_dataset(df_list_fps[3], df_list_text[3])

so in the end we decided to ask mrs. Spencer to pick us out one when went over to get her little girl we heard last week she was going so we sent her word by Richard Spencer's Folks at Carmody to bring us a smart likely boy of about 10 or 11 we decided that would be the best date


In [22]:
print(f.shape)
print(len(l))

(80, 3000)
70


In [23]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")


/home/alien/Programming/env/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [24]:
model.generation_config.language = "english"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None


In [25]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


In [26]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [27]:
import evaluate

metric = evaluate.load("wer")

In [28]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


In [29]:
# from transformers import Seq2SeqTrainingArguments

# training_args = Seq2SeqTrainingArguments(
#     output_dir="./whisper-small-en-sep28",  # change to a repo name of your choice
#     per_device_train_batch_size=16,
#     gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
#     learning_rate=1e-5,
#     warmup_steps=500,
#     max_steps=5000,
#     gradient_checkpointing=True,
#     fp16=True,
#     evaluation_strategy="steps",
#     per_device_eval_batch_size=8,
#     predict_with_generate=True,
#     generation_max_length=225,
#     save_steps=1000,
#     eval_steps=1000,
#     logging_steps=25,
#     report_to=["tensorboard"],
#     load_best_model_at_end=True,
#     metric_for_best_model="wer",
#     greater_is_better=False,
#     push_to_hub=True,
# )


# from transformers import Seq2SeqTrainingArguments

# training_args = Seq2SeqTrainingArguments(
#     output_dir="./whisper-small-sep28",
#     per_device_train_batch_size=16,
#     gradient_accumulation_steps=1,
#     learning_rate=1e-5,
#     warmup_steps=500,
#     max_steps=5910,  # Adjusted for 30 epochs
#     gradient_checkpointing=True,
#     fp16=True,
#     evaluation_strategy="steps",
#     per_device_eval_batch_size=8,
#     predict_with_generate=True,
#     generation_max_length=225,
#     save_steps=1000,
#     eval_steps=1000,
#     logging_steps=25,
#     report_to=["tensorboard"],
#     load_best_model_at_end=True,
#     metric_for_best_model="wer",
#     greater_is_better=False,
#     push_to_hub=True,
#     num_train_epochs=30  # Specify the number of epochs
# )


from transformers import Seq2SeqTrainingArguments

# Calculate the new max_steps for 50 epochs
# Assuming 5910 steps correspond to 30 epochs, steps per epoch = 5910 / 30 = 197 steps/epoch
steps_per_epoch = 5910 / 30
max_steps_50_epochs = int(steps_per_epoch * 50)

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-sep28",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=max_steps_50_epochs,  # Adjusted for 50 epochs
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)


In [30]:
from sklearn.model_selection import train_test_split

# Assuming `batch` is the list of processed samples (from the previous code)
# Perform an 80/20 train/test split
train_batch, test_batch = train_test_split(batch, test_size=0.2, random_state=42)

# `train_batch` will contain 80% of the samples
# `test_batch` will contain 20% of the samples

# Now, you can pass `train_batch` and `test_batch` to your data collator or model


In [31]:
print(len(train_batch))
print(len(test_batch))

3130
783


In [32]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_batch,
    eval_dataset=test_batch,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)


/home/alien/Programming/env/lib/python3.12/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'Repository' (from 'huggingface_hub.repository') is deprecated and will be removed from version '1.0'. Please prefer the http-based alternatives instead. Given its large adoption in legacy code, the complete removal is only planned on next major release.
For more details, please read https://huggingface.co/docs/huggingface_hub/concepts/git_vs_http.
  warnings.warn(warning_message, FutureWarning)
Cloning https://huggingface.co/justanotherinternetguy/whisper-small-sep28 into local empty directory.
Download file pytorch_model.bin:   0%|          | 8.00k/922M [00:00<?, ?B/s]




















Download file pytorch_model.bin:   0%|          | 1.17M/922M [00:01<13:17, 1.21MB/s]






























































Download file pytorch_model.bin:   1%|          | 5.62M/922M [00:03<07:47, 2.05MB/s]





Download file pytorch_model.bin:   1%|          | 

In [33]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
trainer.train()

/home/alien/Programming/env/lib/python3.12/site-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
  0%|          | 0/9850 [00:00<?, ?it/s]/home/alien/Programming/env/lib/python3.12/site-packages/transformers/trainer.py:2610: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)
/home/alien/Programming/env/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default 

{'loss': 0.9705, 'learning_rate': 4.6000000000000004e-07, 'epoch': 0.06}


  1%|          | 50/9850 [00:40<2:13:45,  1.22it/s]

{'loss': 0.9264, 'learning_rate': 9.600000000000001e-07, 'epoch': 0.13}


  1%|          | 75/9850 [01:01<2:15:38,  1.20it/s]

...

100%|██████████| 9850/9850 [2:33:25<00:00,  1.22it/s]/home/alien/Programming/env/lib/python3.12/site-packages/transformers/trainer.py:2172: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimenta

{'loss': 0.0001, 'learning_rate': 4.2780748663101606e-09, 'epoch': 25.13}


100%|██████████| 9850/9850 [2:33:26<00:00,  1.07it/s]

{'train_runtime': 9206.6622, 'train_samples_per_second': 8.559, 'train_steps_per_second': 1.07, 'train_loss': 0.04394488546834209, 'epoch': 25.13}


TrainOutput(global_step=9850, training_loss=0.04394488546834209, metrics={'train_runtime': 9206.6622, 'train_samples_per_second': 8.559, 'train_steps_per_second': 1.07, 'train_loss': 0.04394488546834209, 'epoch': 25.13})

In [34]:
kwargs = {
    "dataset": "SEP-28K",  # a 'pretty' name for the training dataset
    "dataset_args": "config: en, split: test",
    "language": "en",
    "model_name": "Whisper-Small Augmented for SEP-28k",  # a 'pretty' name for your model
    "finetuned_from": "openai/whisper-small",
    "tasks": "automatic-speech-recognition",
}


In [35]:
trainer.push_to_hub(**kwargs)

Upload file runs/Feb16_11-02-04_alien/events.out.tfevents.1739722216.alien.7591.0:   0%|          | 1.00/69.1k [00:00<?, ?B/s]To https://huggingface.co/justanotherinternetguy/whisper-small-sep28
   96e9c80..f6620ce  main -> main

Upload file runs/Feb16_11-02-04_alien/events.out.tfevents.1739722216.alien.7591.0: 100%|██████████| 69.1k/69.1k [00:02<00:00, 35.3kB/s]
To https://huggingface.co/justanotherinternetguy/whisper-small-sep28
   f6620ce..2a489cf  main -> main



'https://huggingface.co/justanotherinternetguy/whisper-small-sep28/commit/f6620cef3ca70efc151c7d1e4cce2706c0e2714f'